In [ ]:
from astropy.cosmology import Planck18
#from zdm import cosmology as cos
from zdm.zdm import misc_functions
from zdm.zdm import parameters
#from zdm import survey
#from zdm import pcosmic
#from zdm import iteration as it
from zdm.zdm import loading
#from zdm import io
import matplotlib.pyplot as plt
from zdm.zdm import figures

import numpy as np
#from matplotlib import pyplot as plt
from pkg_resources import resource_filename
#import time

from frb.dm import igm
from frb.scripts.pzdm_mag import main as pzdm_mag_main
import argparse
import pandas as pd
from scipy.stats import spearmanr

In [ ]:
#params 
nmcs = 1000 
nu = 1. # GHz
t_samp = 1.182  # ms
bandwidth = 1.0 # MHz
snrthreshold = 10. # SNR threshold
mean_DM_MW = 80. # pc cm^-3
disp_DM_MW = 50. # pc cm^-3


plot_chime_grid = True # whether to plot the CHIME pzdm grid 

survey_path ='/Users/lmasriba/FRBs/zdm/zdm/data/Surveys/CHIME/' # this needs to be changed to the path where the survey files are located on your machine

In [ ]:
from networkx import sigma

from frb.scripts.dm_z import DM




def set_state():
    """
    Initializes the state without the contribution from the host DM - uses best fit parameters from Hoffman+2025.
    """

    param_dict={'lmean': 0.01, 'lsigma': 0.42,  'Wlogmean': -1,'WNbins': 1,'Wlogsigma': 0.1, 'Slogmean': -2,'Slogsigma': 0.1, # these essentially turn off DM host and set all FRB widths to ~0 (or close enough)
        # next the parameters that control the FRB population and cosmology
        'sfr_n': 0.21, 'alpha': 0.11,'lEmax': 41.37, 'lEmin': 39.47, 'gamma': -1.04, 
        'H0': 70.23, 'halo_method': 0, 'sigmaDMG': 0.0, 'sigmaHalo': 0.0,
        'lC': -7.61, 'min_lat': 0.0}
    
    state = parameters.State()
    state.set_astropy_cosmo(Planck18)
    state.update_params(param_dict)

    # Initialize with state parameters
    #cos.set_cosmology(state.params)
    #cos.init_dist_measures()
    return state


def creategrid(plotgrid=False):
    """ creates the grid for the CHIME survey - this is done by loading the six 
    declination bins separately and then summing over them to get the total rates, 
    reps, and singles for the full CHIME survey. """
    # defines CHIME grids to load
    NDECBINS=6
    cnames=[]
    for i in np.arange(NDECBINS):
        cname="CHIME_decbin_"+str(i)+"_of_6"
        cnames.append(cname)
    survey_dir = survey_path

    state = set_state()

    css,cgs = loading.surveys_and_grids(survey_names=cnames, init_state=state, rand_DMG=False,sdir = survey_dir, repeaters=True)

    # compiles sums over all six declination bins
    crates = cgs[0].rates * 10**cgs[0].state.FRBdemo.lC * css[0].TOBS
    creps = cgs[0].exact_reps * cgs[0].state.rep.RC
    csingles = cgs[0].exact_singles * cgs[0].state.rep.RC
        
    for i,g in enumerate(cgs):
            s = css[i]
            if i ==0:
                continue
            else:
                crates += g.rates * 10**g.state.FRBdemo.lC * s.TOBS
                creps += g.exact_reps * g.state.rep.RC
                csingles += g.exact_singles * g.state.rep.RC

    # make sure it rewrites cgs[0].rates (also for reps and singles) to the sum over all six declination bins
    cgs[0].rates = crates 
    cgs[0].exact_reps = creps
    cgs[0].exact_singles = csingles

    if plotgrid:
         figures.plot_grid(
                    cgs[0].rates,
                    cgs[0].zvals,
                    cgs[0].dmvals,
                    norm=3,
                    #logrange=5,
                    log=True,
                    project=False,
                    showplot=True,
                    save=False,
                    zmax=2.5,
                    DMmax=2500,
                    Aconts=[0.01, 0.1, 0.5] #,
                    #cont_clrs=[1,1,1]
                    )

    return cgs[0]


def create_frbs(NMC):
    """ creates NMC FRBs by sampling the CHIME grid and then simulating DM host and 
    tau host for each FRB, as well as the corresponding SNR accounting for observational biases. 
    The results are saved to a .npz file. 
    
    
    The DM host - tau relation is based on the MW pulsar relation from Faber et al. 2024 https://arxiv.org/pdf/2405.14182, which is given by:
    """
    # get the grid
    g = creategrid(plotgrid=plot_chime_grid)
    

    # create NMC DM_cosmic data points by sampling the grid
    frbs = g.GenMCSample(NMC)
    frbs = np.array(frbs)
    zs = frbs[:,0]
    DMcos = frbs[:,1]
    snrs = frbs[:,3]
    #bs = frbs[:,2]
    #ws = frbs[:,4]


    # create NMC  DM host and corresponding tau via the MW_PSR relation  (at the host frame)
    # For DM host - tau 
    #  τ (DM, ν)]mw,psr = 1.90 × 10−7 ms × ν−α DM^1.5 × (1 + 3.55 × 10−5 DM^3) from the refs in Faber et al. 2024 https://arxiv.org/pdf/2405.14182
    # sigma_logtau = 0.76
    dm_hosts = 10**np.random.normal(loc=1.8, scale=0.6, size=NMC)
    tau_host = 10**np.random.normal(loc=np.log10(1.9e-7 * nu**-4 * dm_hosts**1.5 * (1 + 3.55e-5*dm_hosts**3)), scale=0.76) # ms

        # DM MW
    dm_MW = np.random.normal(loc=mean_DM_MW,scale=disp_DM_MW, size=NMC)
    dm_MW[np.where(dm_MW < 10.)[0]] = 20. # pc cm^-3

    # DM smearing
    DM_smear = 8.3 * bandwidth * (dm_hosts/(1+zs)+DMcos+ dm_MW) * nu**-3.0 *1e-3 # ms

    # true SNR accounting for observational biases   
    true_snr = snrs * ((t_samp**2 + DM_smear**2)/(t_samp**2 + DM_smear**2 + (tau_host*3./(1.+zs)**3.)**2))**0.25

    snrs = true_snr

    # total DM 
    dm_frb = DMcos + dm_hosts/(1+zs) + dm_MW   

    # DMcosmic from Macquart et al. 2020
    DMcos_macquart = [igm.average_DM(zs[i]).value for i in range(len(zs))] # pc cm^-3

    # estimated DM host
    dm_host_estimated = dm_frb - DMcos_macquart - mean_DM_MW

    # save the results
    np.savez('tau_dm_sims_chime.npz', dm_host_est=dm_host_estimated, tau_host=tau_host, zs=zs, DMcos_macquart=DMcos_macquart, dm_frb=dm_frb, dm_MW=dm_MW,DMcos=DMcos, snrs=snrs, dm_hosts=dm_hosts)


def create_frbs_c22(NMC):
    """ creates NMC FRBs by sampling the CHIME grid and then simulating DM host and
    tau host for each FRB, as well as the corresponding SNR accounting for observational biases.
    The results are saved to a .npz file.
    
    The DM host - tau relation is based on the Cordes+22 relation https://iopscience.iop.org/article/10.3847/1538-4357/ac6873/pdf, which is given by:
    tau_host = 48 ms * AFG * (DM_host/1000)^2
    where AFG is a factor that accounts for the unknown distribution of scattering material in the host 
    
    """
    # get the grid
    g = creategrid(plotgrid=plot_chime_grid)

    # create DM_EG by sampling the grid
    frbs = g.GenMCSample(NMC)
    frbs = np.array(frbs)
    zs = frbs[:,0]
    DMcos = frbs[:,1]
    snrs = frbs[:,3]
    #bs = frbs[:,2]
    #ws = frbs[:,4]


    # create NMC  DM host and corresponding tau via the cordes+22  relation  (at the host frame)
    dm_hosts = 10**np.random.normal(loc=1.8, scale=0.6, size=NMC)
    afg = np.random.uniform(0.01, 10, NMC)
    tau_host = 48.*1e-3 * afg * dm_hosts**2 /1000.    # ms


    # DM MW
    dm_MW = np.random.normal(loc=mean_DM_MW,scale=disp_DM_MW, size=NMC)
    dm_MW[np.where(dm_MW < 10.)[0]] = 20. # pc cm^-3

    # DM smearing
    DM_smear = 8.3 * bandwidth * (dm_hosts/(1+zs)+DMcos+ dm_MW) * nu**-3.0 *1e-3 # ms

    # true SNR accounting for observational biases   
    true_snr = snrs * ((t_samp**2 + DM_smear**2)/(t_samp**2 + DM_smear**2 + (tau_host*3./(1.+zs)**3.)**2))**0.25
    snrs = true_snr

     # total DM 
    dm_frb = DMcos + dm_hosts/(1+zs) + dm_MW   


    arra =np.load('dm_Mcquart.npz') # in the function above I calculated this using the igm module 
                                    # from frb, but here I load it from a file to save time since it doesn't change between the two simulations. 
                                    # just run it once and save it to a file, then load it here.
    dm_Mcq = arra['dm']
    z = arra['z']

    # DMcosmic from Macquart et al. 2020
    DMcos_macquart = np.interp(zs, z, dm_Mcq) # pc cm^-3

    # estimated DM host
    dm_host_estimated = dm_frb - DMcos_macquart - mean_DM_MW

    # save the results
    np.savez('tau_dm_sims_chime_c22.npz', dm_host_est=dm_host_estimated, tau_host=tau_host, zs=zs, DMcos_macquart=DMcos_macquart, dm_frb=dm_frb, dm_MW=dm_MW,DMcos=DMcos, snrs=snrs, dm_hosts=dm_hosts)

# load the data
def load_data(file_path):
    """
    # Load the data from the npz file"
    """
    # Check if the file exists
    if not os.path.exists(file_path):
        raise FileNotFoundError(f"The file {file_path} does not exist.")
    # Load the data
    data = np.load(file_path)
    dm_host = data['dm_host_est']    # estimated DM host in the OBSERVER frame
    tau_host = data['tau_host']      # tau host in the HOST frame
    zs = data['zs']
    DMcos_macquart = data['DMcos_macquart']
    dm_frb = data['dm_frb']
    dm_MW = data['dm_MW']
    DMcos = data['DMcos']
    snrs = data['snrs']
    dm_hosts_sampled = data['dm_hosts']
    return dm_host, tau_host, zs, DMcos_macquart, dm_frb, dm_MW, DMcos, snrs, dm_hosts_sampled


def plot_dm_tau(snr_th, nobj=None, redshift=False):
    """
    # Create a log-log plot of dm_host and tau_host"
    """
    # Load the data
    file_path = data 
    dm_host_estimated, tau_host, zs, DMcos_macquart, dm_frb, dm_MW, DMcos, snrs, dm_hosts_sampled = load_data(file_path)
    # Filter the data based on the SNR threshold
    dm_host_estimated = dm_host_estimated[np.where(snrs > snr_th)]
    tau_host = tau_host[np.where(snrs > snr_th)]         
    dm_frb = dm_frb[np.where(snrs > snr_th)]
    zs = zs[np.where(snrs > snr_th)]


    # make a subset of tau_host
    if nobj is not None and nobj < len(tau_host):
        # Randomly select nobj samples
        indices = np.random.choice(len(tau_host), size=nobj, replace=False)
        tau_host = tau_host[indices]
        dm_frb = dm_frb[indices]
        dm_host_estimated = dm_host_estimated[indices]
        zs = zs[indices]

    # Assert that dm_host_estimated and tau_host do not contain NaN values
    assert not np.isnan(dm_host_estimated).any(), "dm_host_estimated contains NaN values"
    assert not np.isnan(tau_host).any(), "tau_host contains NaN values"

    plt.figure(figsize=(8, 6))
    #plt.loglog(dm_host_estimated, tau_host, 'o', markersize=5, alpha=0.7)

    if redshift==True:
        # assume the redshift is not known so we move the tau_host to the observed frame 
        # and then back to the host frame with the inferred redshift
        tau_host = tau_host *3. / (1 + zs)**3 # ms

        #compute the redshifts
        zmins = np.zeros(len(dm_frb))
        zmaxs = np.zeros(len(dm_frb))
        z50s = np.zeros(len(dm_frb))
        for i in range(len(dm_frb)):
            zmin, zmax, z50 = redshift_estimate(dm_frb[i])
            zmins[i] = zmin
            zmaxs[i] = zmax
            z50s[i] = z50

        zmins[zmins < 0] = 0
        zmaxs[zmaxs < 0] = 0
        z50s[z50s < 0] = 0

        tau_l = tau_host /3. * (1 + zmins)**3
        tau_u = tau_host /3. * (1 + zmaxs)**3
        tau_med = tau_host /3. * (1 + z50s)**3


        dm_l = dm_host_estimated * (1 + zmins)
        dm_u = dm_host_estimated * (1 + zmaxs)
        dm_med = dm_host_estimated * (1 + z50s)
        # Add error bars to the plot
        #dm_l[dm_l < 0] = 0
        #dm_med[dm_med < 0] = 0
        #dm_l[dm_med < 0] = 0
        plt.errorbar(dm_med, tau_med, xerr=[np.abs(dm_med - dm_l), np.abs(dm_u - dm_med)], yerr=[tau_med - tau_l, tau_u - tau_med], fmt='.', markersize=5, alpha=0.7, label='${\\rm SNR}>$'+str(snr_th))

    else:
        sc = plt.scatter(dm_host_estimated*(1.+zs), tau_host, c=zs, cmap='viridis', s=40, alpha=0.7, label='${\\rm SNR}>$'+str(snr_th))
        cbar = plt.colorbar(sc)
        cbar.set_label('$z$', fontsize=15)




    plt.vlines(500,1e-7,1e5, color='gray', linestyle=':', linewidth=1, alpha=0.5)

    plt.yscale('log')
    plt.xscale('symlog', linthresh=500, linscale=3)
    # Plot the theoretical line for comparison
    dm_line = np.logspace(0, 4, 100)
    tau_line = 1.9e-7 * dm_line**1.5 * (1 + 3.55e-5 * dm_line**3)
    tau_up = 10**(np.log10(1.9e-7 * dm_line**1.5 * (1 + 3.55e-5 * dm_line**3)) + 0.76)
    tau_dw = 10**(np.log10(1.9e-7 * dm_line**1.5 * (1 + 3.55e-5 * dm_line**3)) - 0.76)
    plt.plot(dm_line, tau_line, '-', color='k') #, label='$\\tau$(DM) Cordes et al. 2022')
    plt.plot(dm_line, tau_up, '--', color='k') #, label='$\\sigma_{\\log\\tau}$')
    plt.plot(dm_line, tau_dw, '--', color='k')
    plt.xlabel('DM$_{\\rm host}^{\prime}$ (pc cm$^{-3}$)', fontsize=18)
    plt.ylabel('$\\tau_{\\rm host}$ (ms) at 1 GHz', fontsize=18)
    plt.xlim(-200, 5000)
    plt.ylim(1e-7, 1e5)
    #plt.title('Log-Log Plot of DM Host vs Tau Host', fontsize=14)
    plt.grid(True, which="both", linestyle='--', linewidth=0.5)
    plt.xticks(fontsize=17)
    plt.yticks(fontsize=17)
    plt.legend(fontsize=17)
    plt.tight_layout()
    plt.savefig('./dm_host_est_tau_host_'+str(snr_th)+'_100.png' , dpi=300)
    plt.show()

def dm_tau_corr_coeff(N, snr_th, nobj):
    """
    # Calculate an array with the correlation coefficient between dm_host and tau_host
    """
    # output array
    out = np.empty(N, dtype=float)
    mean_dm = np.zeros(N, dtype=float)

    # Load the data
    file_path = data
    dm_host_estimated, tau_host, zs, DMcos_macquart, dm_frb, dm_MW, DMcos, snrs, dm_hosts_sampled = load_data(file_path)
    # Filter the data based on the SNR threshold
    dm_host_estimated = dm_host_estimated[np.where(snrs > snr_th)]
    tau_host = tau_host[np.where(snrs > snr_th)]        
    zs = zs[np.where(snrs > snr_th)]



    # make a subset of tau_host
    assert nobj < len(tau_host)

    for n in range(N):
        # Randomly select nobj samples
        np.random.seed()
        indices = np.random.choice(len(tau_host), size=nobj)
        #print (f"Indices: {indices}")
        tau_host_sam = tau_host[indices]
        dm_host_sam = dm_host_estimated[indices]
        zs_sam = zs[indices]

        # Calculate the correlation coefficient
        dms =np.log10(dm_host_sam *(1+zs_sam))
        nonan_indices = np.isfinite(dms) 
        taus = np.log10(tau_host_sam[nonan_indices])
        dms = dms[nonan_indices]
        corr_coeff, p_value = spearmanr(dms, taus)
        #print(f"Correlation coefficient between DM host and tau host: {corr_coeff}")
        out[n] = corr_coeff
        mean_dm[n] = np.mean(dm_host_sam*(1+zs_sam))

    # Create the main histogram plot
    plt.figure(figsize=(8, 6))
    plt.hist(out, bins=30, color='cornflowerblue', alpha=0.6, edgecolor='black')

    # Calculate percentiles
    p16, p50, p84 = np.nanpercentile(out, [16, 50, 84])

    # Add vertical lines for percentiles
    plt.axvline(p16, color='red', linestyle='--', label=f'16th: {p16:.3f}')
    plt.axvline(p50, color='green', linestyle='-', label=f'50th (median): {p50:.3f}')
    plt.axvline(p84, color='orange', linestyle='--', label=f'84th: {p84:.3f}')

    # Add text to the top right of the plot
    plt.text(0.25, 0.95, f'SNR > {snr_th}\n $N_{{\\rm FRB}}$ = {nobj}', 
             transform=plt.gca().transAxes, 
             fontsize=14, 
             verticalalignment='top', 
             horizontalalignment='right', 
             bbox=dict(facecolor='white', alpha=0.8, edgecolor='gray'))

    # Add labels and legend
    plt.xlabel('$r_{xy}$', fontsize=18)
    plt.ylabel('Counts', fontsize=18)
    plt.xticks(fontsize=17)
    plt.yticks(fontsize=15)
    #plt.grid(True, linestyle='--', linewidth=0.5)
    plt.xlim(-1, 1)
    plt.tight_layout()
    plt.legend(fontsize=16, loc='lower right')

    # Create an inset plot
    # Add a shaded background to the inset
    # Add a frame around the inset and paint the background with a faint color
    inset_ax = plt.axes([0.1095, 0.35, 0.4, 0.4], facecolor='whitesmoke', box_aspect=1)
    inset_ax.hist2d(mean_dm, out, bins=(50, 50), cmap='Blues', cmin=1, norm=plt.matplotlib.colors.LogNorm())
    inset_ax.set_xlabel('$\\overline{\\rm DM}_{\\rm host}$', fontsize=12)
    inset_ax.set_ylabel('$r_{xy}$', fontsize=12)
    inset_ax.set_yticks([-1, -0.5, 0, 0.5, 1])
    inset_ax.set_yticklabels([-1, -0.5, 0, 0.5, 1], fontsize=8)
    inset_ax.yaxis.tick_right()
    inset_ax.yaxis.set_label_position("right")
    inset_ax.spines['left'].set_visible(False)
    inset_ax.spines['right'].set_visible(True)
    inset_ax.set_xlim(0, 2500)
    inset_ax.set_ylim(-1, 1)
    inset_ax.tick_params(axis='both', which='major', labelsize=8)
    inset_ax.grid(True, linestyle='--', linewidth=0.5)
    
    
    # Save the plot
    plt.savefig('./dm_tau_corr_coeff_snr'+str(snr_th)+'_nobj'+str(nobj)+'_spearman_c22.png', dpi=300)
    # Show the plot
    plt.show()


    return


def corr_z_dm(N, snr_th, nobj, zth=None, dmth=None):
    """
    # Calculate an array with the correlation coefficient between dm_host and tau_host
    """
    # output array
    out = np.empty(N, dtype=float)
    mean_dm = np.zeros(N, dtype=float)

    # Load the data
    file_path = data
    dm_host_estimated, tau_host, zs, DMcos_macquart, dm_frb, dm_MW, DMcos, snrs, dm_hosts_sampled = load_data(file_path)
    # Filter the data based on the SNR threshold
    dm_host_estimated = dm_host_estimated[np.where(snrs > snr_th)]
    dm_frb = dm_frb[np.where(snrs > snr_th)]
    tau_host = tau_host[np.where(snrs > snr_th)]        
    zs = zs[np.where(snrs > snr_th)]


    # If z and dm are provided, filter the data based on them
    if zth is not None:
        # Filter based on redshift
        indices = np.where(zs <= zth)[0]
        dm_host_estimated = dm_host_estimated[indices]
        dm_frb = dm_frb[indices]
        tau_host = tau_host[indices]
        zs = zs[indices]
    if dmth is not None:
        # Filter based on DM
        indices = np.where(dm_frb <= dmth )[0]
        dm_host_estimated = dm_host_estimated[indices]
        dm_frb = dm_frb[indices]
        tau_host = tau_host[indices]
        zs = zs[indices]

    print (f"Number of samples after filtering: {len(tau_host)}")
    if len(tau_host) <432:
        return np.nan, np.nan, np.nan

    # make a subset of tau_host
    assert nobj < len(tau_host)

    for n in range(N):
        # Randomly select nobj samples
        np.random.seed()
        indices = np.random.choice(len(tau_host), size=nobj)
        #print (f"Indices: {indices}")
        tau_host_sam = tau_host[indices]
        dm_host_sam = dm_host_estimated[indices]
        zs_sam = zs[indices]

        # Calculate the correlation coefficient
        dmx =np.log10(dm_host_sam *(1+zs_sam))
        nonan_indices = np.isfinite(dmx) 
        taus = np.log10(tau_host_sam[nonan_indices])
        dmx = dmx[nonan_indices]
        corr_coeff , p_value = spearmanr(dmx, taus)
        #print(f"Correlation coefficient between DM host and tau host: {corr_coeff}")
        out[n] = corr_coeff
        #mean_dm[n] = np.mean(dm_host_sam*(1+zs_sam))


    # Calculate percentiles
    p16, p50, p84 = np.nanpercentile(out, [16, 50, 84])
    return p16, p50, p84



def run_tau_corr_zdm(N, snr_th, nobj, zs=None, dms=None):
    """
    # Run the correlation calculation and plot the results
    """

    # Create the main histogram plot
    plt.figure(figsize=(8, 6))

    # Add text to the lower right corner of the plot
    plt.text(0.95, 0.05, f'$N_{{\\rm FRB}}$ = {nobj}', 
             transform=plt.gca().transAxes, 
             fontsize=14, 
             verticalalignment='bottom', 
             horizontalalignment='right', 
             bbox=dict(facecolor='white', alpha=0.8, edgecolor='gray'))

    
    plt.ylabel('$r_{xy}$', fontsize=18)
    plt.xticks(fontsize=16)                 
    plt.yticks(fontsize=16)
    plt.grid(True, linestyle='--', linewidth=0.5)
    # Check if zs or dms are provided
    colors = ['blue', 'orange']
    for x,snr_th in enumerate([0,10]):  


        if zs is not None:
            var = 'z'
            p16s = np.zeros(len(zs))
            p50s = np.zeros(len(zs))
            p84s = np.zeros(len(zs))
            for i, zz in enumerate(zs):
                p16s[i], p50s[i], p84s[i] = corr_z_dm(N, snr_th, nobj, zth=zz)

        if dms is not None:
            var = 'dm'
            p16s = np.zeros(len(dms))
            p50s = np.zeros(len(dms))
            p84s = np.zeros(len(dms))
            for i, dm in enumerate(dms):
                p16s[i], p50s[i], p84s[i] = corr_z_dm(N, snr_th, nobj, dmth=dm)



        if var == 'z':
            plt.xscale('log')
            plt.xlim(0.017, 2.1)
            plt.xlabel('$z$', fontsize=18)
            plt.xticks([0.02, 0.04, 0.07, 0.1, 0.2, 0.4, 0.7, 1.0, 2.0],[ '0.02', '0.04', '0.07', '0.1', '0.2', '0.4', '0.7', '1.0', '2.0'], fontsize=16)
            plt.plot(zs, p50s, 'o-', color= colors[x], label='50th (Median) SNR>'+str(snr_th))
            plt.fill_between(zs, p16s, p84s, alpha=0.2, label='16th-84th  SNR>'+str(snr_th), color=colors[x])
            plt.ylim(0, 1)

        elif var == 'dm':
            plt.xlim(150, 5200)
            plt.xscale('log')
            plt.xlabel('${\\rm DM_{FRB}\,(pc\\,cm^{-3})}$',fontsize=18)
            plt.xticks([200,300,500,700,1000,2000,4000],[str(x) for x in [200,300,500,700,1000,2000,4000]], fontsize=16)
            plt.plot(dms, p50s, 'o-', color= colors[x], label='50th (Median) SNR>'+str(snr_th))
            plt.fill_between(dms, p16s, p84s,  alpha=0.2, label='16th-84th  SNR>'+str(snr_th), color=colors[x])
            plt.ylim(-0.1, 1)
        #plt.xlim(0, 5)
    
    plt.legend(fontsize=16,loc='upper left')
    plt.tight_layout()
    plt.hlines(0.,150,5200, color='k', linestyle='dotted')
    plt.savefig('./dm_tau_corr_zdm_snr'+str(snr_th)+'_nobj'+str(nobj)+'_'+str(var)+'_spearman_c22.png', dpi=300)
    # Show the plot
    plt.show()

In [ ]:
def main():

    # these two below can be run once to create grids and save them, then just load the .npz files to save time
    create_frbs(25000) # this creates the simulated FRB population using the DM host - tau relation from Faber et al. 2024 https://arxiv.org/pdf/2405.14182 (pulsars)
    #create_frbs_c22(25000) # this creates the simulated FRB population using the DM host - tau relation from Cordes+22 https://iopscience.iop.org/article/10.3847/1538-4357/ac6873/pdf

    # load the data  -- to not create the grid every time (choose one of the two simulations to load)
    #data = 'tau_dm_sims_chime.npz'
    #data = 'tau_dm_sims_chime_c22.npz' 

    # plot the DM host - tau relation for the simulated FRBs
    plot_dm_tau(snr_th=0, nobj=100, redshift=False)

    # calculate the correlation coefficient between DM host and tau host 
    dm_tau_corr_coeff(N=100000,snr_th=0,nobj=100)

    # calculate the correlation coefficient between DM host and tau host as a function of redshift and DM
    run_tau_corr_zdm(N=10000, snr_th=10, nobj=100, zs=[0.02, 0.04, 0.07, 0.1, 0.2, 0.4, 0.7, 1.0, 2.0]) #
    run_tau_corr_zdm(N=10000, snr_th=10, nobj=100, dms=np.logspace(2.2, 3.7, 10))   


main()